# M7 — MIMIC-IV Temporal Stratification (Manuscript, Supervisor Feedback Item A-4)

**Purpose.** Test whether the fixed GP formula's calibration drifts over admission year within MIMIC-IV,
adding a temporal validation dimension at near-zero cost. Addresses supervisor feedback item A-4 (W5).

**Preliminary check (already done, 2026-09-16).** MIMIC-IV v3.1 applies a per-patient random date shift
for de-identification, so absolute calendar years are not recoverable, but the `patients` table's
`anchor_year_group` field bins each patient into a three-year admission window. Verified: all 6,152
cohort patients matched to a non-null `anchor_year_group` (100% match via `subject_id`), split into 5
groups (2008-2010 through 2020-2022), smallest group n=658 with 207 deaths — well above the
underpowered threshold (n<100 or events<15) specified for this analysis. No contingency needed.

**Plan.**
1. Load the already-computed MIMIC-IV GP predictions (`mimic_gp_predictions.csv` — no retraining,
   no recalibration, the same fixed eICU-derived formula used throughout the paper).
2. Join to `mimic_cohort.parquet` (stay_id -> subject_id) then to the raw MIMIC-IV `patients` table
   (subject_id -> anchor_year_group).
3. Group by anchor_year_group. Report n, mortality %, AUROC, ECE per group.
4. Flag any group with n<100 or deaths<15 as underpowered (none expected, per preliminary check).
5. Save a supplementary table: Year Group, n, Mortality %, AUROC, ECE.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

PROJECT = Path(r"C:\ML PROJECT\sepsis-gp")
sys.path.insert(0, str(PROJECT))
from src.metrics import compute_ece

OUT_DIR = PROJECT / "results" / "manuscript" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIMIC_PATIENTS_CSV = Path(r"C:\ML PROJECT\DATASETS\mimic-iv-3.1\hosp\patients.csv.gz")

MIN_N = 100
MIN_EVENTS = 15

## Step 1 — Load predictions and join to anchor_year_group

In [2]:
preds = pd.read_csv(PROJECT / "data" / "processed" / "mimic_gp_predictions.csv")
cohort = pd.read_parquet(PROJECT / "data" / "processed" / "mimic_cohort.parquet")[["subject_id", "stay_id"]]
patients = pd.read_csv(MIMIC_PATIENTS_CSV, usecols=["subject_id", "anchor_year_group"])

df = preds.merge(cohort, on="stay_id", how="left").merge(patients, on="subject_id", how="left")

print(f"Total predictions: {len(preds)}")
print(f"Matched to subject_id: {df['subject_id'].notna().sum()}")
print(f"Matched to anchor_year_group: {df['anchor_year_group'].notna().sum()}")
assert df['anchor_year_group'].notna().all(), "Unmatched rows found — investigate before proceeding"
print("\nGroups:", sorted(df['anchor_year_group'].unique()))

Total predictions: 6152
Matched to subject_id: 6152
Matched to anchor_year_group: 6152

Groups: ['2008 - 2010', '2011 - 2013', '2014 - 2016', '2017 - 2019', '2020 - 2022']


## Step 2 — Per-group AUROC and ECE, using the fixed formula's existing predictions (no retraining)

In [3]:
records = []
for grp, sub in df.groupby("anchor_year_group", sort=True):
    y = sub["hospital_expire_flag"].to_numpy(dtype=np.float64)
    p = sub["gp_prob"].to_numpy(dtype=np.float64)
    n = len(sub)
    deaths = int(y.sum())
    underpowered = (n < MIN_N) or (deaths < MIN_EVENTS)

    auroc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan
    ece = compute_ece(y, p)

    records.append({
        "year_group": grp,
        "n": n,
        "deaths": deaths,
        "mortality_pct": round(100 * deaths / n, 2),
        "auroc": round(auroc, 4) if not np.isnan(auroc) else None,
        "ece": round(ece, 4),
        "underpowered": underpowered,
    })

temporal = pd.DataFrame(records)
temporal

,year_group,n,deaths,mortality_pct,auroc,ece,underpowered
0,2008 - 2010,957,241,25.18,0.6725,0.0867,False
1,2011 - 2013,658,207,31.46,0.6943,0.1477,False
2,2014 - 2016,1319,367,27.82,0.7027,0.1014,False
3,2017 - 2019,1773,529,29.84,0.7243,0.1238,False
4,2020 - 2022,1445,478,33.08,0.7292,0.1642,False


## Step 3 — Save output

In [4]:
out_path = OUT_DIR / "M7_mimic_temporal_stratification.csv"
temporal.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

print(f"\nOverall MIMIC-IV (n={len(df)}): AUROC={roc_auc_score(df['hospital_expire_flag'], df['gp_prob']):.4f}, "
      f"ECE={compute_ece(df['hospital_expire_flag'].to_numpy(dtype=float), df['gp_prob'].to_numpy(dtype=float)):.4f}")
print("(for comparison against published overall MIMIC-IV numbers: AUROC 0.709, ECE 0.125)")

Saved: C:\ML PROJECT\sepsis-gp\results\manuscript\tables\M7_mimic_temporal_stratification.csv

Overall MIMIC-IV (n=6152): AUROC=0.7087, ECE=0.1245
(for comparison against published overall MIMIC-IV numbers: AUROC 0.709, ECE 0.125)


## Findings

The sanity check confirmed pipeline validity: the pooled-sample AUROC (0.7087) and ECE (0.1245) computed
here from the per-patient predictions matched the published overall MIMIC-IV numbers (0.709, 0.125)
almost exactly.

All 6,152 cohort patients matched to a non-null `anchor_year_group` (100% match), yielding five
well-powered admission-year groups (smallest n=658, 207 deaths), none flagged as underpowered.

Discrimination did not degrade over the study period; if anything it improved slightly and fairly
steadily with more recent admission years: AUROC rose from 0.6725 (2008-2010) to 0.6943 (2011-2013),
0.7027 (2014-2016), 0.7243 (2017-2019), and 0.7292 (2020-2022) -- a monotonic increase across all five
groups. This should not be over-interpreted as evidence the fixed formula "learned" anything about later
years (it is a fixed, unretrained expression); it more plausibly reflects secular changes in casemix,
coding practice, or ICU care over the 2008-2022 window rather than temporal robustness per se, and the
diagnostic does not separate these explanations.

Calibration error tracked mortality prevalence far more clearly than it tracked calendar time. Group
mortality prevalence ranged from 25.18% to 33.08%, all substantially above the eICU training prevalence
of 16.88%. When groups are ordered by their prevalence gap from 16.88% rather than chronologically, ECE
increases essentially monotonically with that gap:

| Group | Mortality | Gap from eICU prevalence | ECE |
|---|---|---|---|
| 2008-2010 | 25.18% | 8.30 pp | 0.0867 |
| 2014-2016 | 27.82% | 10.94 pp | 0.1014 |
| 2017-2019 | 29.84% | 12.96 pp | 0.1238 |
| 2011-2013 | 31.46% | 14.58 pp | 0.1477 |
| 2020-2022 | 33.08% | 16.20 pp | 0.1642 |

This is a clean, additional line of evidence for the paper's central calibration finding: MIMIC-IV
miscalibration is prevalence-driven, not merely a generic cross-database or temporal-drift effect. The
group with the largest prevalence gap (2020-2022) has the worst calibration error of any group,
notwithstanding having the best discrimination of any group -- reinforcing the paper's broader point
(Sections 4.6-4.7) that discrimination and calibration transport independently, and that prevalence,
not time, is the dominant driver of calibration failure here.

Overall: temporal stratification does not reveal a distinct "calibration drift over time" problem
requiring its own correction; the temporal pattern in ECE is adequately explained by the same
prevalence-shift mechanism already identified and addressed by the paper's prevalence-adaptive
recalibration method (Section 4.7).